### 14/15 Puzzle und Varianten
Der initilale Zustand ist


```python
board = ((0, 1, 2, 3), 
         (4, 5, 6, 7), 
         (8, 9, 10, 11), 
         (12, 13, 14, 15),
         )
```
wobei die Null das Loch darstellt. 
Ein legaler Zug ist ein Positionstausch der Null mit einem ihrer Nachbarn.

Eine Folge von Züge ist ein Weg der Null übers Brett. Die Zugfolge  
`[(0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]`  
schiebt die Null nach rechts unten, und der neue Spielzustand ist dann
```python
((4, 1, 2, 3), 
 (8, 5, 6, 7), 
 (12, 9, 10, 11), 
 (13, 14, 15, 0),
 )
```
Ziel ist es, zu einem gegebenen Zustand einen Weg der Null zu finden,
der den initialen Zustand wieder herstellt.


**Nur die Hälfte aller Zustände sind durch Verschieben der Null erreichbar**  
Stellen wir uns vor, die Felder des Bretts sind schachbrettartig gefärbt und das
Feld links oben sei weiss.
Bei jedem Zug wechselt die Null die Feldfarbe.
Immer nach einer geraden Anzahl Zügen ist die Null auf einem weissen Feld.
Eine Rundreise der Null ist also immer eine gerade Anzahl Züge lang.
Da ein Zug einem Positionstausch entspricht, lassen sich so höchstens
Zustände realisieren, welche durch eine gerade Anzahl von Positionstauschen erreichbar sind.

Man kann zeigen, dass genau die Hälfte aller Zustände 
erreichbar sind. Bei einem Brett mit $n$ Feldern sind dies $(n(n-1)(n-2)\cdots1)/2 = n!/2$ Zustände.


Bei einem  3x3 Brett ist diese Anzahl mit 181'440 noch durchsuchbar,
bei einem 4x4 Brett mit 10'461'394'944'000 Zuständen ist die nicht mehr möglich.

Um trotzdem einen Weg zum initialen Zustand zu finden,
versuchen wir Folgendes: Aus unserer Liste `nodes_to_visit`
entfernen wir den Knoten, bei welchem 
die Summe der [Manhatten-Distanzen](https://de.wikipedia.org/wiki/Manhattan-Metrik) der
Zahlen zu ihren Ursprungsfelder minimal ist. Wir wählen also den Knoten, der in diesem Sinne am
nächsten beim initialen Zustand liegt.

In [ ]:
import matrix_helpers as M


def as_tuple(mat):
    return tuple(tuple(row) for row in mat)


def as_list(mat):
    return [list(row) for row in mat]


def swap(matrix, p, q):
    matrix[p[1]][p[0]], matrix[q[1]][q[0]] = matrix[q[1]][q[0]], matrix[p[1]][p[0]]


def get_board(ncol, nrow):
    n = ncol * nrow
    board = tuple(tuple(range(i, i+ncol)) for i in range(0, n, ncol))
    return board


def show_board(board):
    for row in board:
        print(row)
    print(40*'-')

In [ ]:
board = get_board(4, 4)
show_board(board)

In [ ]:
board

In [ ]:
hole = M.find(board, 0)
hole

***
Ein Nachbar eines Knotens `(hole, board)` ist ein Tuple `(new_hole, new_board)`.
`new_hole` ist die neue Position der Null, `new_board` der neue Spielzustand.
***

In [ ]:
def get_neighbors(hole, board):
    for new_pos in M.get_neighbors(board, hole):
        new_board = as_list(board)
        swap(new_board, hole, new_pos)
        yield new_pos, as_tuple(new_board)

In [ ]:
ns = get_neighbors(hole, board)

In [ ]:
print(next(ns))

***
Um einen zufälligen Zustand zu erhalten,
schicken wir die Null auf einen zufälligen Weg. Jeder Zug
soll dabei zu einem neuen Zustand führen.
***

In [ ]:
import random


def scramble(board, n=200):
    seen = {board}
    hole = M.find(board, 0)

    for _ in range(n):
        neighbors = list(get_neighbors(hole, board))
        random.shuffle(neighbors)

        for hole, board in neighbors:
            if board not in seen:
                seen.add(board)
                break

    return hole, board

In [ ]:
hole, board = scramble(get_board(3, 3), 2)
show_board(board)

### Aufgaben
- Verwende depth-first search, um einen Weg zum initialen Zustand bei einem
3x3 Brett zu finden.  
- Verwende breadth-first search, um einen Weg zum initialen Zustand bei einem
3x3 Brett zu finden.  
- Verwende breadth-first search, um herauszufinden, wieviele Zustände erreichbar sind.

Das Ziel, das wir suchen, ist jeweils der initiale Zustand  
`goal = get_board(*M.get_dims(board))`.  

Ein Knoten ist ein Tuple `(hole, board)`. Im `go_back` Dict speichern wir
Key-Value Paare der Form  
`{new_board: (hole, board), ...}`.


Interessiert sind wir am Rückweg der Null.
```python
def get_path_home(node, go_back):
    '''Weg der Null'''
    path = []
    while node:  # Knoten hat die Form (hole, board) oder None
        path.append(node[0])
        node = go_back[node[-1]]

    return path
```

Zudem wollen wir eine Funktion 
```python
def follow_path(board, path):
    '''Verschiebe die Null im board entlang path und gib das
       resultierende board zurueck
    ''' 
    ...
```

In [ ]:
from collections import deque


def get_path_home(node, go_back):
    '''Weg der Null'''
    path = []
    while node:  # Knoten hat die Form (hole, board) oder None
        path.append(node[0])
        node = go_back[node[-1]]

    return path


def follow_path(board, path):
    if len(path) == 0:
        return board

    hole = M.find(board, 0)
    if hole != path[0]:
        raise ValueError(f'Bad path. Path must start at {hole}.')

    board = as_list(board)
    n = len(path)
    for i in range(1, n):
        M.swap(board, path[i-1], path[i])
    return as_tuple(board)


def search_df(board, get_neighbors):
    goal = get_board(*M.get_dims(board))
    hole = M.find(board, 0)
    node = (hole, board)
    count = 0

    nodes_to_visit = [node]
    go_back = {board: None}

    while nodes_to_visit:
        count += 1
        (hole, board) = nodes_to_visit.pop()
        if board == goal:
            print(f'Success! Count: {count}')
            return (hole, board), go_back

        for new_hole, new_board in get_neighbors(hole, board):
            if new_board in go_back:
                continue

            go_back[new_board] = hole, board
            nodes_to_visit.append((new_hole, new_board))

    print(f'Failure! Count: {count}')
    return (hole, board), go_back



def search_bf(board, get_neighbors):
    goal = get_board(*M.get_dims(board))
    hole = M.find(board, 0)
    node = (hole, board)
    count = 0

    nodes_to_visit = deque([node])
    go_back = {board: None}

    while nodes_to_visit:
        count += 1
        (hole, board) = nodes_to_visit.pop()
        if board == goal:
            print(f'Success! Count: {count}')
            return (hole, board), go_back

        for new_hole, new_board in get_neighbors(hole, board):
            if new_board in go_back:
                continue

            go_back[new_board] = hole, board
            nodes_to_visit.appendleft((new_hole, new_board))

    print(f'Failure! Count: {count}')
    return (hole, board), go_back

In [ ]:
hole, board = scramble(get_board(3, 3), 200)
show_board(board)

In [ ]:
board

In [ ]:
(h, b), go_back = search_df(board, get_neighbors)
path = get_path_home((h, b), go_back)
len(path), path[::-1][:5]

In [ ]:
follow_path(board, path[::-1])

In [ ]:
(h, b), go_back = search_bf(board, get_neighbors)
path = get_path_home((h, b), go_back)
len(path), path[::-1][:5]

In [ ]:
follow_path(board, path[::-1])

In [ ]:
# Kommentiere in search_bf Folgendes aus, damit alle Zustaende besucht werden
#     if board == goal:
#         print(f'Success! Count: {count}')
#         return (hole, board), go_back

(h, b), go_back = search_bf(board, get_neighbors)
len(go_back)  # 181440

In [ ]:
# 

### Aufgabe
Verwende nun eine Greedy-Search mit einem Priority-Queue.
Im Priority-Queue sind Tuple der Form `(priority, hole, board)`.  
Die `priority` ist dabei ein Tuple `(manhatten_dist(board), counter)`, wobei
```python
def manhatten_dist(board):
    tot = 0
    ncol = len(board[0])

    for r, vals in enumerate(board):
        for c, v in enumerate(vals):
            row, col = divmod(v, ncol)
            tot += abs(col - c) + abs(row - r)
    return tot
```

In [ ]:
import heapq


def manhatten_dist(board):
    tot = 0
    ncol = len(board[0])

    for r, vals in enumerate(board):
        for c, v in enumerate(vals):
            row, col = divmod(v, ncol)
            tot += abs(col - c) + abs(row - r)
    return tot



def search_greedy(board, get_neighbors, get_priority):
    goal = get_board(*M.get_dims(board))
    hole = M.find(board, 0)

    count = 0
    priority = (get_priority(board), count)
    nodes_to_visit = [(priority, hole, board)]
    go_back = {board: None}

    while nodes_to_visit:
        _, hole, board = heapq.heappop(nodes_to_visit)
        if board == goal:
            print(f'Success! count: {count}')
            return (hole, board), go_back

        for new_hole, new_board in get_neighbors(hole, board):
            if new_board in go_back:
                continue

            go_back[new_board] = hole, board
            count += 1
            priority = (get_priority(new_board), count)
            heapq.heappush(nodes_to_visit, (priority, new_hole, new_board))

    print(f'failure: {count}')

In [ ]:
hole, board = scramble(get_board(4, 4), 200)
show_board(board)

In [ ]:
manhatten_dist(board)

In [ ]:
(hole, b), go_back = search_greedy(board, get_neighbors, manhatten_dist)
path = get_path_home((hole, b), go_back)
len(path), path[::-1][:5]

In [ ]:
follow_path(board, path[::-1])